### En este código se hace el ajuste de un perceprón multicapa con operaciones algebraicas en tensorflow. Además para el perceptrón multicapa se hace el ajuste también con keras. Además al final vemos cómo se puede salvar un modelo y sus pesos para podere cargarlos en otra sesión.

In [1]:
import tensorflow as tf
tf.__version__

'2.11.0'

Let's play around with some linear algebra examples

In [2]:
#Álgebra con tensor flow, los vectores son un caso particular de tensor

a = tf.constant([1., 2., 2., 3.])
print(a)


b = tf.constant([4., 5., 5., 6.])
print(b)


c = a + b
print(c)

tf.Tensor([1. 2. 2. 3.], shape=(4,), dtype=float32)
tf.Tensor([4. 5. 5. 6.], shape=(4,), dtype=float32)
tf.Tensor([5. 7. 7. 9.], shape=(4,), dtype=float32)


In [3]:
#producto interior
a = tf.constant([1., 2., 2., 3.])
print(a)


b = tf.constant([4., 5., 5., 6.])
print(b)


c = tf.tensordot(a, b, axes=1)
print(c)

tf.Tensor([1. 2. 2. 3.], shape=(4,), dtype=float32)
tf.Tensor([4. 5. 5. 6.], shape=(4,), dtype=float32)
tf.Tensor(42.0, shape=(), dtype=float32)


### Neural Network

In [12]:
#Perceptrón multicapa con dos capas

import tensorflow as tf
tf.__version__

import numpy as np
data = np.array(
    [
        [100,35,35,12,0.32],
        [101,46,35,21,0.34],
        [130,56,46,3412,12.42],
        [131,58,48,3542,13.43]
    ]
)

#Normalizar los inputs, que son tres, y el output
x = data[:,1:-1] #selecciona desde la segunda hasta la penultima columna, es decir, es una matriz 
x = x / np.linalg.norm(x)#se normaliza la matriz
y_target = data[:,-1] #selecciona la última columna (-1) de todas las filas (:) del array data. Esta es la variable dependiente.
y_target = y_target / np.linalg.norm(y_target) 

#Pesos de la capa 1, por la manera en que está definido, tenemos los pesos iniciales
#de cada uno de los tres inputs y la capa oculta, estamos suponiendo que esta tiene
#tres nodos
w1 = tf.Variable([[1,1,1],[1,1,1],[1,1,1]],dtype=tf.float64)
#Los pesos de la capa oculta a la capa de salida (un solo output) son 3 y valen 1
w2 = tf.Variable([1,1,1],dtype=tf.float64)

#La capa 1 tiene una función de activación sigmoide
def layer1(x):
    return tf.sigmoid(tf.tensordot(x,w1,axes=1))

print(layer1(x))

#La capa 2 también tiene función de activación sigmoid, ver que depende de los valores que se obtienen
#en la capa 1
def layer2(x):
    return tf.sigmoid(tf.tensordot(layer1(x),w2,axes=1))


#Fijar el tipo de optimizador y la función de pérdida, en este caso el ECM en escala logarítmica
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01) #Adaptative Moment Estimaton
loss_object = tf.keras.losses.MeanSquaredLogarithmicError()

#porporcionar input y output, ver que partimos de la capa 2, predicted, esta necesita de la capa 1, así
#que se hace este proceso de back propagation
def train_step(x, y):
    with tf.GradientTape() as tape:
        predicted = layer2(x)   
        loss_value = loss_object(y, predicted)
        print(loss_value)
        grads = tape.gradient(loss_value, [w1,w2])
        optimizer.apply_gradients(zip(grads, [w1,w2]))
    
def train(epochs):
    for epoch in range(epochs):
            train_step(x, y_target)
    print ('Epoch {} finished'.format(epoch))
    
train(epochs = 1000)



tf.Tensor(
[[0.50416671 0.50416671 0.50416671]
 [0.50518291 0.50518291 0.50518291]
 [0.67133984 0.67133984 0.67133984]
 [0.67732112 0.67732112 0.67732112]], shape=(4, 3), dtype=float64)
tf.Tensor(0.17364808023981926, shape=(), dtype=float64)
tf.Tensor(0.17278880870339167, shape=(), dtype=float64)
tf.Tensor(0.171922643201505, shape=(), dtype=float64)
tf.Tensor(0.17104962963873255, shape=(), dtype=float64)
tf.Tensor(0.17016977983076914, shape=(), dtype=float64)
tf.Tensor(0.16928310231047636, shape=(), dtype=float64)
tf.Tensor(0.16838960919042362, shape=(), dtype=float64)
tf.Tensor(0.16748931879981982, shape=(), dtype=float64)
tf.Tensor(0.16658225709823027, shape=(), dtype=float64)
tf.Tensor(0.1656684586202982, shape=(), dtype=float64)
tf.Tensor(0.1647479671976912, shape=(), dtype=float64)
tf.Tensor(0.16382083655260868, shape=(), dtype=float64)
tf.Tensor(0.16288713080274125, shape=(), dtype=float64)
tf.Tensor(0.1619469248954222, shape=(), dtype=float64)
tf.Tensor(0.16100030497886197, shap

In [13]:
#pesos primera capa
w1 

<tf.Variable 'Variable:0' shape=(3, 3) dtype=float64, numpy=
array([[ 0.44021503,  0.44021503,  0.44021503],
       [ 0.70261203,  0.70261203,  0.70261203],
       [-6.31300169, -6.31300169, -6.31300169]])>

In [14]:
layer1(x)

<tf.Tensor: shape=(4, 3), dtype=float64, numpy=
array([[0.49818302, 0.49818302, 0.49818302],
       [0.49554206, 0.49554206, 0.49554206],
       [0.01253514, 0.01253514, 0.01253514],
       [0.01063458, 0.01063458, 0.01063458]])>

In [15]:
x

array([[0.00711406, 0.00711406, 0.0024391 ],
       [0.0093499 , 0.00711406, 0.00426843],
       [0.01138249, 0.0093499 , 0.6935188 ],
       [0.01178901, 0.00975642, 0.71994243]])

### Keras

In [5]:
#proceso análogo al anterior, pero con keras
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.utils import to_categorical

data = np.array(
    [
        [100,35,35,12,0],
        [101,46,35,21,0],
        [130,56,46,3412,1],
        [131,58,48,3542,1]
    ]
)

x = data[:,1:-1]
y_target = data[:,-1]

x = x / np.linalg.norm(x)

#Usamos capas densas, todas las conexiones entre cada nodode uno nodo y la capa siguiente existen
#Hay 3 inputs (lo que está en input_shape lo indica) y tres nodos en la capa oculta (lo que viene después
#de Dense lo indica), hay un solo output
model = Sequential()
model.add(Dense(3, input_shape=(3,), activation='sigmoid'))
model.add(Dense(1, activation='sigmoid'))

#El output es binario, así que usamos binary cross entropy como función de pérdida, queremos como métrica
#solo accuracy
model.compile(optimizer=SGD(learning_rate=0.1),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(x, y_target, epochs=1000,
          verbose=1)

Epoch 1/1000
1/1 [==============================] - 1s 510ms/step - loss: 0.6721 - accuracy: 0.5000
Epoch 2/1000
1/1 [==============================] - 0s 5ms/step - loss: 0.6716 - accuracy: 0.5000
Epoch 3/1000
1/1 [==============================] - 0s 10ms/step - loss: 0.6710 - accuracy: 0.5000
Epoch 4/1000
1/1 [==============================] - 0s 5ms/step - loss: 0.6705 - accuracy: 0.5000
Epoch 5/1000
1/1 [==============================] - 0s 8ms/step - loss: 0.6700 - accuracy: 0.5000
Epoch 6/1000
1/1 [==============================] - 0s 6ms/step - loss: 0.6695 - accuracy: 0.5000
Epoch 7/1000
1/1 [==============================] - 0s 7ms/step - loss: 0.6690 - accuracy: 0.5000
Epoch 8/1000
1/1 [==============================] - 0s 7ms/step - loss: 0.6685 - accuracy: 0.5000
Epoch 9/1000
1/1 [==============================] - 0s 6ms/step - loss: 0.6680 - accuracy: 0.5000
Epoch 10/1000
1/1 [==============================] - 0s 7ms/step - loss: 0.6675 - accuracy: 0.5000
Epoch 11/1000
1/

In [6]:
#Predicciones bajo el modelo, según el cutoff elegido ya lo mandamos a la categoría cero o 1
model.predict(x)

1/1 [==============================] - 0s 51ms/step


array([[0.06900769],
       [0.06968533],
       [0.93728465],
       [0.94561654]], dtype=float32)

In [7]:
# Save model (tanto la arquitectura como los pesos)

model.save('model_1.h5')

In [8]:
# Save model as JSON  (JavaScript Object Notation, it is a lightweight format for storing and transporting data)
#and weights as HDF5 (hierarchical data format versión 5)

#import json

#json_string = model.to_json() # model.to_yaml()
#json_file.write(json_string)
#model.save_weights('weights.h5')
#json_string

# serialize model to JSON
model_json = model.to_json()
with open("model_1_only.json", "w") as json_file:
        json_file.write(model_json)
# serialize weights to HDF5
model.save_weights('weights_only.h5')
print("Saved model to disk")
print(model_json)

Saved model to disk
{"class_name": "Sequential", "config": {"name": "sequential", "layers": [{"class_name": "InputLayer", "config": {"batch_input_shape": [null, 3], "dtype": "float32", "sparse": false, "ragged": false, "name": "dense_input"}}, {"class_name": "Dense", "config": {"name": "dense", "trainable": true, "dtype": "float32", "batch_input_shape": [null, 3], "units": 3, "activation": "sigmoid", "use_bias": true, "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": null}}, "bias_initializer": {"class_name": "Zeros", "config": {}}, "kernel_regularizer": null, "bias_regularizer": null, "activity_regularizer": null, "kernel_constraint": null, "bias_constraint": null}}, {"class_name": "Dense", "config": {"name": "dense_1", "trainable": true, "dtype": "float32", "units": 1, "activation": "sigmoid", "use_bias": true, "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": null}}, "bias_initializer": {"class_name": "Zeros", "config": {}}, "kernel_regu

In [1]:
#Para probar, cerramos este notebook y regresamos (aunque no es necesario)

# Load from JSON and set weights
import tensorflow as tf

model = tf.keras.models.load_model('model_1.h5')
model

In [6]:
#Predicciones bajo el modelo, según el cutoff elegido ya lo mandamos a la categoría cero o 1
#Se supone debieran ser nuevos datos, pero usamos los mismo que teníamos para verificar los
#resultados

import numpy as np

data = np.array(
    [
        [100,35,35,12,0],
        [101,46,35,21,0],
        [130,56,46,3412,1],
        [131,58,48,3542,1]
    ]
)

x = data[:,1:-1]
x = x / np.linalg.norm(x)

model.predict(x)

1/1 [==============================] - 0s 52ms/step


array([[0.06900769],
       [0.06968533],
       [0.93728465],
       [0.94561654]], dtype=float32)

In [7]:
#Cargar los pesos y arquitectura por separado

#load json and create model

import tensorflow as tf
from keras.models import model_from_json

json_file = open('model_1_only.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
print(loaded_model)

In [8]:
# load weights into new model

#from keras.models import model_from_json
#import json

#model = tf.keras.models.model_from_json(json_string)
#model.load_weights('weights.h5')
#print(model)

loaded_model.load_weights("weights_only.h5")
print("Loaded model from disk")
print(loaded_model)

Loaded model from disk


In [9]:
#Predicciones bajo el modelo, según el cutoff elegido ya lo mandamos a la categoría cero o 1
#Se supone debieran ser nuevos datos, pero usamos los mismo que teníamos para verificar los
#resultados

import numpy as np

data = np.array(
    [
        [100,35,35,12,0],
        [101,46,35,21,0],
        [130,56,46,3412,1],
        [131,58,48,3542,1]
    ]
)

x = data[:,1:-1]
x = x / np.linalg.norm(x)

loaded_model.predict(x)

1/1 [==============================] - 0s 50ms/step


array([[0.06900769],
       [0.06968533],
       [0.93728465],
       [0.94561654]], dtype=float32)